# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'pandas'

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
df = pd.read_csv('data/AviationData.csv', encoding='latin1', low_memory=False)
print(f"Shape: {df.shape}")
df.info()
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
display(pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}))
display(df.describe())
df.head()

Shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make               

,missing_count,missing_pct
Schedule,76307,85.8
Air.carrier,72241,81.3
FAR.Description,56866,64.0
Aircraft.Category,56602,63.7
Longitude,54516,61.3
Latitude,54507,61.3
Airport.Code,38757,43.6
Airport.Name,36185,40.7
Broad.phase.of.flight,27165,30.6
Publication.Date,13771,15.5


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'] >= '1983-01-01']
df = df[df['Investigation.Type'] == 'Accident']
df = df[df['Amateur.Built'] == 'No']
print(df.shape)

(73286, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
# Injury count columns needed to estimate total passengers per flight
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Assumption: NaN in these columns means 0 people in that category were recorded,
# not "unknown" -- so we fill with 0 before summing
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers on the flight as the sum of all four injury/outcome categories
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Drop rows where no passenger total could be established -- can't compute a fraction for these
df = df[df['Total.Passengers'] > 0]

# Derived metric: fraction of passengers who were seriously or fatally injured
# This is the key safety measure the client wants to compare across makes/models
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(df.shape)
df['Fatal.Serious.Fraction'].describe()

(72852, 33)


count    72852.000000
mean         0.286128
std          0.434153
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Fatal.Serious.Fraction, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Check the raw categories in Aircraft.damage
print(df['Aircraft.damage'].value_counts(dropna=False))

# Drop rows where damage status is missing or unknown -- can't classify destruction for these
df = df[df['Aircraft.damage'].isin(['Substantial', 'Destroyed', 'Minor'])]

# Derived metric: binary flag for whether the aircraft was a total loss
# 1 = Destroyed, 0 = Substantial or Minor damage
df['Is.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(df.shape)
df['Is.Destroyed'].value_counts(normalize=True)

Aircraft.damage
Substantial    55401
Destroyed      15375
NaN             1350
Minor            647
Unknown           79
Name: count, dtype: int64
(71423, 34)


Is.Destroyed
0    0.784733
1    0.215267
Name: proportion, dtype: float64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

# identifying cleaning tasks 
Standardization of text values

Converted all entries to uppercase and stripped whitespace (.strip().str.upper()).

This reduced the number of unique values from 2,198 → 1,784, since duplicates like Cessna / CESSNA / cessna collapsed into one.

Handling missing data

Dropped 16 rows where no manufacturer (Make) was recorded.

Filtering by frequency

Retained only manufacturers with ≥50 records to ensure statistical reliability in later comparisons.

This kept 79 makes and 65,175 rows out of 71,407 total rows (~91%).

In [ ]:
# Drop rows with no Make recorded
df = df.dropna(subset=['Make'])

# Standardize casing/whitespace so the same manufacturer isn't split
# across multiple spellings (e.g. "Cessna" vs "CESSNA")
df['Make'] = df['Make'].str.strip().str.upper()

# Keep only makes with a reasonable sample size (threshold: 50)
# so comparisons between makes are statistically meaningful
make_counts = df['Make'].value_counts()
frequent_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(frequent_makes)]

print(df.shape)
print(f'{len(frequent_makes)} makes retained out of {len(make_counts)}')

(65175, 34)
79 makes retained out of 1784


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Drop the few rows with no Model recorded
df = df.dropna(subset=['Model'])

# Standardize casing/whitespace, same reasoning as with Make
df['Model'] = df['Model'].str.strip().str.upper()

# Check: is a Model label unique to one Make, or shared across makes?
models_per_make_count = df.groupby('Model')['Make'].nunique()
shared_models = (models_per_make_count > 1).sum()
print(f'{shared_models} of {len(models_per_make_count)} model labels are shared across more than one Make')

# Since Model isn't unique on its own, combine Make + Model into one identifier per plane type
df['Make.Model'] = df['Make'] + ' ' + df['Model']
print(df['Make.Model'].nunique(), 'unique plane types')

363 of 5091 model labels are shared across more than one Make
5558 unique plane types


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# Merge duplicate/inconsistent category labels found by inspecting value_counts()
df['Engine.Type'] = df['Engine.Type'].replace('UNK', 'Unknown')
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': 'Unknown', 'Unk': 'Unknown'})
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Air Race show', 'Air Race/show')

for col in ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight', 'Number.of.Engines']:
    print(f'--- {col}')
    print(df[col].value_counts(dropna=False))
    print()

--- Engine.Type
Engine.Type
Reciprocating    54155
NaN               3633
Turbo Shaft       2851
Turbo Prop        2404
Unknown           1150
Turbo Fan          721
Turbo Jet          247
NONE                 1
Name: count, dtype: int64

--- Weather.Condition
Weather.Condition
VMC        57590
IMC         4911
NaN         1944
Unknown      717
Name: count, dtype: int64

--- Purpose.of.flight
Purpose.of.flight
Personal                     36303
Instructional                 9214
Aerial Application            4115
Unknown                       3983
Business                      3277
NaN                           2808
Positioning                   1357
Other Work Use                1016
Aerial Observation             689
Ferry                          617
Public Aircraft                610
Executive/corporate            383
Skydiving                      170
Flight Test                    144
Banner Tow                      97
External Load                   80
Public Aircraft - Federal 

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Check what percentage of each column is missing
missing_pct = (df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)
print(missing_pct)

# Drop columns missing more than 40% of their data -- too sparse to be reliable
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
print(cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(df.shape)

Airport.Name              39.8
Broad.phase.of.flight     27.0
Publication.Date          17.1
Engine.Type                5.6
Report.Status              5.3
Number.of.Engines          4.7
Purpose.of.flight          4.3
Weather.Condition          3.0
Registration.Number        1.2
Country                    0.3
Total.Serious.Injuries     0.0
Is.Destroyed               0.0
Fatal.Serious.Fraction     0.0
Total.Passengers           0.0
Total.Uninjured            0.0
Total.Minor.Injuries       0.0
Event.Id                   0.0
Total.Fatal.Injuries       0.0
Investigation.Type         0.0
Amateur.Built              0.0
Model                      0.0
Make                       0.0
Aircraft.damage            0.0
Injury.Severity            0.0
Location                   0.0
Event.Date                 0.0
Accident.Number            0.0
Make.Model                 0.0
dtype: float64
[]
(65162, 28)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
df.to_csv('data/aviation_data_cleaned.csv', index=False)
print(df.shape)

(65162, 28)
